# GrowthScope — Sprint 01: Data Discovery

**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce` (Google Analytics 4 obfuscated sample e-commerce export)
**Author:** GrowthScope project
**Sprint:** 01 — Project Foundation & Data Discovery
**Status of this notebook:** reproducible; runs in three modes (see below)

> [!IMPORTANT]
> **Honesty contract.** This notebook never invents data. If BigQuery credentials are not
> available it does **not** print plausible-looking numbers — it reports that the analysis is
> blocked, states exactly which query to run, and stops. Any chart produced from synthetic
> teaching data is labelled `SYNTHETIC DEMO — NOT REAL DATA`.

---

## Table of contents

1. [Business context](#1-business-context)
2. [Data source](#2-data-source)
3. [Data access](#3-data-access)
4. [Data overview](#4-data-overview)
5. [Data quality](#5-data-quality)
6. [E-commerce events](#6-e-commerce-events)
7. [Initial visualisations](#7-initial-visualisations)
8. [Preliminary findings](#8-preliminary-findings)
9. [Data limitations](#9-data-limitations)
10. [Next steps](#10-next-steps)

## 1. Business context

### Why are we looking at this data at all?

An online store receives thousands of visits but cannot answer basic growth questions from raw
analytics exports:

| Question the business asks | Analytical artefact required |
|---|---|
| Where exactly do visitors leave the purchase journey? | An **ordered** funnel over session events |
| Which acquisition channels bring *buyers*, not just traffic? | Session/purchase metrics split by source |
| How much potential revenue is lost at each step? | Drop-off rate × observed average order value |
| Do customers come back after the first purchase? | Cohorts keyed on a stable user identifier |

### What this notebook is for

This is the **discovery** step. It does not deliver business answers. It delivers something more
important at this stage: a verified understanding of what the source data actually contains,
which questions it can support, and which it cannot.

That distinction protects every later sprint. A funnel built on a `purchase` event that fires
twice per order produces confident, wrong numbers. Measuring that risk now is cheaper than
explaining it later.

### Definitions used throughout (and why they matter)

- An **event** is a single logged interaction. Events are the atomic grain of GA4.
- A **session** is a contiguous period of user activity. GA4 starts a new session after
  **30 minutes of inactivity** — that rule is documented. Whether a session *additionally*
  restarts at midnight (legacy Universal Analytics behaviour) is **not clearly documented for
  GA4** and is treated here as `[UNRESOLVED]`: `sql/exploration/07_session_feasibility.sql`
  measures it empirically rather than assuming it. The distinction matters because it changes
  how many sessions a single continuous visit produces.
- A **user** here means a `user_pseudo_id` — a pseudonymous, cookie-scoped identifier.
  It is *not* a person. The same person on phone and laptop is two users; the same person
  after clearing cookies is two users.

> [!WARNING]
> Confusing these three grains is the single most common source of wrong e-commerce analytics.
> Counting events and calling them sessions inflates every conversion rate. This notebook keeps
> the grain explicit at every step.

## 2. Data source

### The dataset

Google publishes a **demo GA4 e-commerce export** as a public BigQuery dataset so that anyone can
practise on realistic event data without collecting any.

| Property | Value |
|---|---|
| Dataset reference | `bigquery-public-data.ga4_obfuscated_sample_ecommerce` |
| Table pattern | `events_YYYYMMDD` (one daily shard per date) |
| Contents | GA4 event export of the Google Merchandise Store demo account |
| Documented period | November 2020 – January 2021 |
| Equal access model | Public dataset — readable by any authenticated BigQuery user, no cost for data, only for bytes scanned |
| Official documentation | <https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset> |

### Why GA4 exports are shaped this way

A GA4 BigQuery export is **denormalised on purpose**. Instead of a fixed column per measurement,
GA4 writes a generic `event_params` array of key–value pairs. This lets Google add new parameters
without a schema migration, and it is why every serious GA4 project starts with the same task:
unpacking `event_params`.

Two consequences drive everything downstream:

1. `ga_session_id` — the session identifier — is *not* a column. It lives inside `event_params`.
   Any session metric requires `UNNEST` or a correlated subquery to retrieve it.
2. Because `UNNEST` multiplies rows per match, a careless `JOIN UNNEST(event_params)` fans out the
   data and silently inflates every count. The safe pattern is a correlated scalar subquery:

   ```sql
   (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS ga_session_id
   ```

### Limitations we already know about, before looking at a single row

These are properties of the dataset, documented by Google or inherent to GA4, not bugs:

| Limitation | Consequence for GrowthScope |
|---|---|
| Data is **obfuscated and partially adjusted** by Google | Absolute values are illustrative; treat structure as the reliable part |
| No advertising cost table exists | **CAC, ROAS and ad profit are impossible.** Nothing will be fabricated to fill the gap |
| `traffic_source` describes **first-user acquisition** | It must never be used as per-session attribution |
| Window is ~3 months, in the 2020 holiday period | No seasonality claims, no long-run retention claims |
| `user_pseudo_id` is cookie-scoped | "Users" are device-browser identities, not people |

> [!NOTE]
> Verification legend used in the GrowthScope documentation:
> `[DOCUMENTED]` — stated by official Google documentation ·
> `[VERIFIED — live]` — confirmed by running a query on the dataset ·
> `[UNVERIFIED — needs BigQuery]` — plausible but not confirmed here ·
> `[BLOCKED — no BigQuery access]` — could not be checked in this environment.

### Reference material for this notebook

| Document | What it clarifies |
|---|---|
| `docs/dataset_research.md` | Full source research, schema, attribution limits, open questions |
| `docs/data_dictionary.md` | Field-by-field reference for the export schema |
| `docs/metric_dictionary.md` | Metric definitions, formulas, denominators, validation status |
| `docs/data_quality_audit.md` | Findings register with severity, evidence and mitigation |
| `sql/exploration/README.md` | How to execute the SQL used below |

## 3. Data access

### Cost safety first

BigQuery bills by **bytes scanned**, not by rows returned. Two habits keep exploration cheap:

1. **Restrict the shard window.** `bigquery-public-data.ga4_obfuscated_sample_ecommerce` is a
   wildcard table: `events_*` expands to every daily shard in the dataset. Without a
   `_TABLE_SUFFIX` filter, one query can scan the entire dataset. A three-day window scans three
   shards instead of ~90.
2. **Project only the columns you need.** This dataset contains wide nested columns
   (`items`, `user_properties`, `event_params`). Selecting them when they are not needed is the
   most common way an exploration query becomes expensive.

Additional guardrail: `maximum_bytes_billed` makes BigQuery **fail the job** instead of letting it
run away. This notebook sets it from the environment.

| Setting | Default | Where it comes from |
|---|---|---|
| Project billed for the query | *(required for live mode)* | `GOOGLE_CLOUD_PROJECT` |
| Dataset | `bigquery-public-data.ga4_obfuscated_sample_ecommerce` | `BQ_DATASET` |
| Window start / end | `2020-11-01` / `2020-11-03` | `GA4_START_DATE` / `GA4_END_DATE` |
| Maximum bytes billed | `2 000 000 000` (~2 GB) | `BQ_MAX_BYTES_BILLED` |

### The four execution modes

| Mode | Trigger | Behaviour |
|---|---|---|
| `live` | BigQuery credentials present and a project configured | Every query is executed on the real dataset |
| `sample` | `data/sample/*.csv` exist | Pre-fetched **aggregated** result sets are loaded from disk (never raw user events) |
| `blocked` | neither of the above | The notebook reports what is missing, prints the query that *would* run, and produces no numbers |
| `demo` | `GROWTHSCOPE_NOTEBOOK_MODE=demo` | Runs the full pipeline on clearly labelled **synthetic** teaching data so the analysis code itself can be reviewed |

`demo` mode exists so that a reader without a Google Cloud account can still see the notebook work
end to end. Everything it produces is stamped `SYNTHETIC DEMO — NOT REAL DATA`.

> [!WARNING]
> Never commit raw user-level event extracts. Only small, aggregated result sets belong in
> `data/sample/`, and credentials belong in neither.

In [1]:
# ---------------------------------------------------------------------------
# Environment and imports
# ---------------------------------------------------------------------------
import os
import re
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib

# Plotting.
# `%matplotlib inline` renders figures *into the notebook* and needs no display server,
# so the same notebook behaves identically in JupyterLab, in CI, and headless.
# Forcing a backend with `matplotlib.use("Agg")` would silently suppress every chart,
# which is why it is not done here.
%matplotlib inline
import matplotlib.pyplot as plt  # noqa: E402

try:
    from dotenv import load_dotenv
except ImportError:  # dotenv is optional for the notebook's core function
    def load_dotenv(*args, **kwargs):
        return False

# --- Locate the repository root regardless of the kernel's working directory -----
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "sql").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
SQL_DIR = REPO_ROOT / "sql" / "exploration"
SAMPLE_DIR = REPO_ROOT / "data" / "sample"
SRC_DIR = REPO_ROOT / "src"

load_dotenv(REPO_ROOT / ".env", override=False)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))  # make `growthscope` importable without installing

# --- Project settings -------------------------------------------------------
BQ_DATASET = os.getenv("BQ_DATASET", "bigquery-public-data.ga4_obfuscated_sample_ecommerce")
BQ_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT", "").strip()  # billed project, NOT the data project
START_DATE = os.getenv("GA4_START_DATE", "2020-11-01")
END_DATE = os.getenv("GA4_END_DATE", "2020-11-03")
MAX_BYTES_BILLED = int(os.getenv("BQ_MAX_BYTES_BILLED", "2000000000"))
REQUESTED_MODE = os.getenv("GROWTHSCOPE_NOTEBOOK_MODE", "auto").strip().lower()
EXPORT_SAMPLE = os.getenv("GROWTHSCOPE_EXPORT_SAMPLE", "0") == "1"

# Refuse to build anything that would scan the whole dataset.
assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", START_DATE), START_DATE
assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", END_DATE), END_DATE
assert START_DATE <= END_DATE, "window start must not be after window end"

SUFFIX_START = START_DATE.replace("-", "")
SUFFIX_END = END_DATE.replace("-", "")
WINDOW_TABLES = (
    f"`{BQ_DATASET}.events_*` "
    f"/* _TABLE_SUFFIX BETWEEN '{SUFFIX_START}' AND '{SUFFIX_END}' */"
)


def is_empty(frame) -> bool:
    """True when a frame is absent or has no rows.

    Used throughout to keep the notebook runnable when a query was blocked: a missing
    result is a legitimate state that must not raise inside an analysis cell.
    """
    return not isinstance(frame, pd.DataFrame) or frame.empty


def has_rows(frame) -> bool:
    """Inverse of ``is_empty`` - reads better in chart code."""
    return not is_empty(frame)


def rel(path) -> str:
    """Render a path relative to the repository root.

    Notebook output is committed to git, so it must not contain machine-specific absolute
    paths -- a reader on another computer would see a path that means nothing to them.
    """
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT)) or "."
    except ValueError:
        return Path(path).name


print(f"Repository root  : {REPO_ROOT.name} (all paths below are relative to it)")
print(f"Python           : {sys.version.split()[0]}")
print(f"pandas           : {pd.__version__}")
print(f"Exploration window: {START_DATE} -> {END_DATE}  (shards {SUFFIX_START}..{SUFFIX_END})")

Repository root  : Marketfit (all paths below are relative to it)
Python           : 3.12.10
pandas           : 3.0.6
Exploration window: 2020-11-01 -> 2020-11-03  (shards 20201101..20201103)


The cell above deliberately **asserts** the date format and ordering, and prints the shard window.
Cheap defensive checks like these are what stop an expensive accident: a malformed date silently
becomes part of a `_TABLE_SUFFIX` filter, and a reversed window silently returns nothing while
looking like "no data".

In [2]:
# ---------------------------------------------------------------------------
# Resolve execution mode
# ---------------------------------------------------------------------------
def resolve_mode(requested: str) -> tuple[str, str]:
    """Return (mode, human-readable reason).

    Ordering matters: explicit request > live credentials > on-disk samples > blocked.
    """
    from_sample = sorted(SAMPLE_DIR.glob("*.csv")) if SAMPLE_DIR.exists() else []

    if requested == "demo":
        return "demo", "GROWTHSCOPE_NOTEBOOK_MODE=demo: synthetic teaching data (NOT real data)."
    if requested == "live":
        return "live", "GROWTHSCOPE_NOTEBOOK_MODE=live: forcing live BigQuery execution."
    if requested == "sample":
        return "sample", "GROWTHSCOPE_NOTEBOOK_MODE=sample: forcing on-disk aggregates."
    if requested == "blocked":
        return "blocked", "GROWTHSCOPE_NOTEBOOK_MODE=blocked: live results intentionally withheld."

    credentials_reason = "no BigQuery credentials or project configured"
    try:
        from google.cloud import bigquery  # noqa: F401
        from google.auth import default as google_default

        google_default()  # raises if no Application Default Credentials are available
        if BQ_PROJECT:
            return "live", f"credentials found and GOOGLE_CLOUD_PROJECT={BQ_PROJECT}"
        credentials_reason = "credentials found, but GOOGLE_CLOUD_PROJECT is not set (needed for billing/quota)"
    except Exception as exc:  # noqa: BLE001 - any auth failure means the same thing for us
        credentials_reason = f"{type(exc).__name__}: {exc}"

    if from_sample:
        return "sample", f"{len(from_sample)} aggregate CSV file(s) in data/sample/"
    return "blocked", credentials_reason


MODE, MODE_REASON = resolve_mode(REQUESTED_MODE)
IS_SYNTHETIC = MODE == "demo"

BANNER = {
    "live": "LIVE — querying the real GA4 public dataset via BigQuery.",
    "sample": "SAMPLE — loading previously exported AGGREGATE results from data/sample/.",
    "demo": "SYNTHETIC DEMO — generated teaching data. NOT REAL DATA. For code review only.",
    "blocked": "BLOCKED — no BigQuery access in this environment. No results will be produced.",
}[MODE]

print("=" * 78)
print(f"EXECUTION MODE : {MODE.upper()}")
print(f"REASON         : {MODE_REASON}")
print(f"BANNER         : {BANNER}")
print("=" * 78)

if MODE == "blocked":
    print(textwrap.dedent("""
        What happens now
        ---------------
        Every analysis cell below will report its query, then return an empty result.
        No numbers, charts or conclusions will be produced from this notebook run.
        This is intentional: invented analytics are worse than absent analytics.

        To obtain real results, do ONE of the following:

        Option 1 - BigQuery Sandbox (free, no credit card)
          1. Sign in at https://console.cloud.google.com/bigquery
          2. Note the auto-created project id and export it:
             PowerShell : $env:GOOGLE_CLOUD_PROJECT="your-project-id"
             bash       : export GOOGLE_CLOUD_PROJECT=your-project-id
          3. Authenticate locally:
             pip install google-cloud-bigquery db-dtypes
             gcloud auth application-default login
          4. Re-run this notebook.

        Option 2 - pre-fetched aggregates
          Place the aggregate CSVs produced by sql/exploration/*.sql into data/sample/
          using the documented file names (see data/sample/README.md), then re-run.
        """))

EXECUTION MODE : BLOCKED
REASON         : GROWTHSCOPE_NOTEBOOK_MODE=blocked: live results intentionally withheld.
BANNER         : BLOCKED — no BigQuery access in this environment. No results will be produced.

What happens now
---------------
Every analysis cell below will report its query, then return an empty result.
No numbers, charts or conclusions will be produced from this notebook run.
This is intentional: invented analytics are worse than absent analytics.

To obtain real results, do ONE of the following:

Option 1 - BigQuery Sandbox (free, no credit card)
  1. Sign in at https://console.cloud.google.com/bigquery
  2. Note the auto-created project id and export it:
     PowerShell : $env:GOOGLE_CLOUD_PROJECT="your-project-id"
     bash       : export GOOGLE_CLOUD_PROJECT=your-project-id
  3. Authenticate locally:
     pip install google-cloud-bigquery db-dtypes
     gcloud auth application-default login
  4. Re-run this notebook.

Option 2 - pre-fetched aggregates
  Place the 

In [3]:
# ---------------------------------------------------------------------------
# BigQuery client (created lazily; never required by the notebook structure)
# ---------------------------------------------------------------------------
def make_client():
    """Return a BigQuery client with a bytes-billed guard, or None if unavailable."""
    if MODE != "live":
        return None
    from google.cloud import bigquery

    return bigquery.Client(project=BQ_PROJECT)


CLIENT = make_client()
print("BigQuery client:", CLIENT.project if CLIENT is not None else "not created (mode = %s)" % MODE)

BigQuery client: not created (mode = blocked)


### How this notebook consumes the SQL files

The repository keeps its analytical SQL in `sql/exploration/` so that the same queries can be run
from a notebook, the `bq` command line, or a scheduled job. The notebook does not copy them; it
reads them from disk. That is what makes the queries reviewable in one place and keeps this
notebook honest about *what* it executed.

A `.sql` file may contain several statements (the first is the primary analysis query, later ones
are follow-ups). The helpers below split a file into individual statements and drop comments, so
the same file works in the BigQuery console and programmatically.

In [4]:
# ---------------------------------------------------------------------------
# Query execution helpers
# ---------------------------------------------------------------------------
BLOCK_COMMENT = re.compile(r"/\*.*?\*/", flags=re.DOTALL)
LINE_COMMENT = re.compile(r"--[^\n]*")


def read_sql_statements(path: Path) -> list[str]:
    """Split a .sql file into individually executable statements (comments removed).

    Splitting on ';' is safe for these files because they contain no stored procedures,
    string literals with semicolons, or scripting blocks.
    """
    raw = Path(path).read_text(encoding="utf-8")
    without_comments = BLOCK_COMMENT.sub(" ", raw)
    without_comments = LINE_COMMENT.sub(" ", without_comments)
    return [stmt.strip() for stmt in without_comments.split(";") if stmt.strip()]


def show_query(statement: str, width: int = 96, max_lines: int = 14) -> None:
    """Print the statement in a readable form.

    Long queries are truncated: the full text lives in ``sql/exploration/`` and duplicating
    it into every notebook output would make the file unreadable.
    """
    lines = statement.splitlines()
    shown = lines[:max_lines]
    print("-" * 78)
    for line in shown:
        print(textwrap.fill(line, width=width) if len(line) > width else line)
    if len(lines) > max_lines:
        print(f"... [{len(lines) - max_lines} more lines - full query in sql/exploration/]")
    print("-" * 78)


def run_bigquery(statement: str, label: str) -> pd.DataFrame:
    """Execute `statement` in live mode; otherwise explain why nothing was produced."""
    if CLIENT is None:
        print(f"[{MODE}] Skipping '{label}'. Query that would run:")
        show_query(statement)
        return pd.DataFrame()
    import time

    from google.cloud import bigquery

    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED)
    started = time.perf_counter()
    job = CLIENT.query(statement, job_config=job_config)
    frame = job.to_dataframe()
    elapsed = time.perf_counter() - started
    print(f"[live] '{label}': {len(frame):,} rows in {elapsed:.2f}s | "
          f"scanned {(job.total_bytes_processed or 0) / 1e6:,.1f} MB | "
          f"billed {(job.total_bytes_billed or 0) / 1e6:,.1f} MB")
    return frame


def primary_query(filename: str, index: int = 0) -> str:
    """Return one statement from sql/exploration/<filename>, or '' when unavailable."""
    path = SQL_DIR / filename
    if not path.exists():
        return ""
    statements = read_sql_statements(path)
    return statements[index] if index < len(statements) else ""


def load_or_query(filename: str, label: str, index: int = 0, demo_builder=None) -> pd.DataFrame:
    """Mode-aware retrieval for one analysis step.

    live    -> execute the SQL statement from sql/exploration/
    sample  -> load the matching aggregate CSV from data/sample/
    demo    -> build the labelled synthetic teaching frame
    blocked -> print the query and return an empty frame
    """
    if MODE == "demo" and demo_builder is not None:
        frame = demo_builder()
        print(f"[demo] '{label}': {len(frame):,} SYNTHETIC rows")
        return frame

    if MODE == "sample":
        csv_path = SAMPLE_DIR / f"{Path(filename).stem}.csv"
        if csv_path.exists():
            frame = pd.read_csv(csv_path)
            print(f"[sample] '{label}': {len(frame):,} rows from {csv_path.name}")
            return frame
        print(f"[sample] '{label}': {csv_path.name} not found in data/sample/.")
        return pd.DataFrame()

    statement = primary_query(filename, index=index)
    if MODE == "live" and not statement:
        print(f"[live] '{label}': sql/exploration/{filename} is missing; nothing to run.")
        return pd.DataFrame()
    return run_bigquery(statement, label)


print("Helpers ready. SQL directory:", rel(SQL_DIR), "| exists:", SQL_DIR.exists())

Helpers ready. SQL directory: sql\exploration | exists: True


## 4. Data overview

Four facts must be established before any metric is defined:

1. **Which tables actually exist?** A wildcard table is only useful if the shards are present.
   A missing shard is indistinguishable from "no activity on that day" unless it is checked.
2. **What is the actual date coverage?** The documented period is not the same as the period
   available in every project.
3. **How large is the sample?** Event volume and user volume set the statistical floor: with a
   few thousand events, a 0.2 percentage-point conversion difference between two channels is
   noise, not a finding.
4. **What is the event taxonomy?** Funnel analysis depends on specific event names existing.
   Their absence is a hard blocker; their presence is only a prerequisite.

The queries below are read directly from `sql/exploration/`.

### 4.1 Shard inventory (preflight)

Query: `sql/exploration/00_project_settings.sql` — lists the `events_*` shards that exist inside
the declared window. This is the cheapest useful query in the sprint, and it is the one that turns
"the query returned nothing" into "the query returned nothing *because the shard is absent*".

In [5]:
def demo_shard_inventory() -> pd.DataFrame:
    dates = pd.date_range(START_DATE, END_DATE, freq="D")
    return pd.DataFrame(
        {
            "table_suffix": dates.strftime("%Y%m%d"),
            "table_name": "events_" + dates.strftime("%Y%m%d"),
            "synthetic_note": ["SYNTHETIC DEMO"] * len(dates),
        }
    )


shard_df = load_or_query("00_project_settings.sql", "shard inventory", demo_builder=demo_shard_inventory)
shard_df

[blocked] Skipping 'shard inventory'. Query that would run:
------------------------------------------------------------------------------
SELECT DISTINCT
  _TABLE_SUFFIX AS shard_suffix
FROM
  `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
ORDER BY
  shard_suffix
------------------------------------------------------------------------------


""


**How to read this.** Each row is one daily export shard that exists in the window. If the list is
shorter than the number of days in the window, that day has no export — a *coverage* problem, not a
traffic problem. GA4 stops writing a shard when the property produces no events, and the public
dataset may skip days for other reasons too.

### 4.2 Event taxonomy

Query: `sql/exploration/01_event_inventory.sql` — distinct `event_name` values with event counts,
distinct users and distinct sessions in the window.

> [!NOTE]
> Counts here are **events**, not sessions and not users. An event count answers "does this event
> exist and is it common?" It does not answer "how many people did this?".

In [6]:
# Synthetic teaching data used ONLY in demo mode. Deterministic, clearly labelled.
DEMO_RNG = np.random.default_rng(seed=20201101)

DEMO_EVENTS = {
    "page_view": 0.42,
    "session_start": 0.09,
    "first_visit": 0.03,
    "view_item": 0.14,
    "view_item_list": 0.07,
    "add_to_cart": 0.05,
    "view_cart": 0.04,
    "begin_checkout": 0.03,
    "add_shipping_info": 0.02,
    "add_payment_info": 0.015,
    "purchase": 0.01,
    "scroll": 0.05,
    "user_engagement": 0.015,
}


def demo_events_table(n_events: int = 12_000) -> pd.DataFrame:
    event_weights = np.asarray(list(DEMO_EVENTS.values()), dtype=float)
    event_weights = event_weights / event_weights.sum()  # normalise: floats never sum exactly to 1
    names = DEMO_RNG.choice(list(DEMO_EVENTS), size=n_events, p=event_weights)

    days = pd.date_range(START_DATE, END_DATE, freq="D")
    day_weights = np.asarray([0.30, 0.36, 0.34][: len(days)], dtype=float)
    day_weights = day_weights / day_weights.sum()
    day_index = DEMO_RNG.choice(len(days), size=n_events, p=day_weights)

    user_ids = [f"SYNTH-USER-{i:05d}" for i in DEMO_RNG.integers(0, 4200, size=n_events)]
    session_ids = DEMO_RNG.integers(1_604_000_000, 1_606_000_000, size=n_events)
    return pd.DataFrame(
        {
            "event_name": names,
            "event_date": days[day_index].strftime("%Y-%m-%d"),
            "user_pseudo_id": user_ids,
            "ga_session_id": session_ids,
            "synthetic_note": "SYNTHETIC DEMO",
        }
    )


def demo_event_inventory() -> pd.DataFrame:
    events = demo_events_table()
    grouped = (
        events.groupby("event_name", as_index=False)
        .agg(event_count=("event_name", "size"))
        .sort_values("event_count", ascending=False)
    )
    grouped["event_share_pct"] = (grouped["event_count"] / grouped["event_count"].sum() * 100).round(2)
    funnel_steps = {"view_item", "add_to_cart", "begin_checkout", "purchase", "view_item_list", "view_cart"}
    grouped["is_funnel_step"] = grouped["event_name"].isin(funnel_steps)
    return grouped


event_df = load_or_query("01_event_inventory.sql", "event inventory", demo_builder=demo_event_inventory)
event_df.head(30)

[blocked] Skipping 'event inventory'. Query that would run:
------------------------------------------------------------------------------
WITH window_events AS (
   
   
   
   
   
  SELECT
    event_name
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
),
event_counts AS (
... [22 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


### 4.3 Daily volume

Query: `sql/exploration/02_daily_event_volume.sql` — events per day, distinct users per day, and
events per user per day.

> [!WARNING]
> Summing *distinct users per day* never equals *distinct users over the period*. A user active on
> two days is counted twice. Daily uniques are comparable day to day; they are not additive. The
> additive version is average daily active users (DAU), and it is a different metric.

In [7]:
def demo_daily_volume() -> pd.DataFrame:
    events = demo_events_table()
    daily = (
        events.groupby("event_date", as_index=False)
        .agg(
            event_count=("event_name", "size"),
            distinct_users=("user_pseudo_id", "nunique"),
            distinct_sessions=("ga_session_id", "nunique"),
        )
        .sort_values("event_date")
    )
    daily["events_per_user"] = (daily["event_count"] / daily["distinct_users"]).round(2)
    return daily


daily_df = load_or_query("02_daily_event_volume.sql", "daily event volume", demo_builder=demo_daily_volume)
daily_df

[blocked] Skipping 'daily event volume'. Query that would run:
------------------------------------------------------------------------------
WITH window_events AS (
   
   
  SELECT
    event_date,
    user_pseudo_id
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
),
daily AS (
  SELECT
    event_date,
... [22 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


## 5. Data quality

### 5.1 Missing user identifiers

Query: `sql/exploration/03_user_identification.sql`.

A missing `user_pseudo_id` is **not** data corruption. GA4 cannot assign a pseudonymous identifier
when the user has declined analytics cookies, when consent mode blocks storage, or when browser
privacy features restrict it. Those events still exist and still carry a session id.

What matters is the *rate*: if 30 % of events have no user id, then every user-level metric
(retention, cohorts, repeat purchase) is computed on 70 % of reality, and that number must be
stated next to the metric. Funnel math at session level is less affected — but only if the session
id is still present, which is what section 5.2 checks.

In [8]:
def demo_user_identification() -> pd.DataFrame:
    events = demo_events_table()
    total = len(events)
    # Deliberately introduce a realistic 4% missing pseudo-id rate so the quality checks have
    # something to find in demo mode (synthetic only).
    missing_mask = DEMO_RNG.random(total) < 0.04
    identified = total - int(missing_mask.sum())
    return pd.DataFrame(
        {
            "metric": [
                "total_events",
                "events_missing_user_pseudo_id",
                "share_missing_user_pseudo_id_pct",
                "distinct_user_pseudo_id",
                "events_with_nonempty_user_id_pct",
                "share_missing_first_touch_ts_pct",
            ],
            "value": [
                float(total),
                float(missing_mask.sum()),
                round(missing_mask.mean() * 100, 3),
                float(events["user_pseudo_id"].nunique()),
                round((1 - float((events["user_pseudo_id"].str.len() == 0).mean())) * 100, 3),
                None,  # not modelled in synthetic data on purpose
            ],
            "synthetic_note": "SYNTHETIC DEMO",
        }
    )


user_df = load_or_query("03_user_identification.sql", "user identification", demo_builder=demo_user_identification)
user_df

[blocked] Skipping 'user identification'. Query that would run:
------------------------------------------------------------------------------
WITH window_events AS (
   
   
  SELECT
    user_pseudo_id,
    user_id,
    user_first_touch_timestamp
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
)
SELECT
  COUNT(*) AS total_events,
... [28 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


### 5.2 Session identification feasibility

Query: `sql/exploration/07_session_feasibility.sql`.

The critical structural question of the whole project. GA4 stores `ga_session_id` as a
**UNIX timestamp of session start in seconds**, not as a global sequence. Two users who start a
session in the same second therefore share the same `ga_session_id` value.

That means:

- `ga_session_id` alone is **not unique** and must never be used as a session key.
- The correct session grain is the composite key **`(user_pseudo_id, ga_session_id)`**.
- Consequences of getting this wrong: sessions collapse across users, session counts are
  understated, and every per-session conversion rate is inflated.

The query pair below measures both facts: how many session values map to more than one user, and
the resulting events-per-session distribution once the composite key is used.

In [9]:
def demo_session_feasibility() -> pd.DataFrame:
    events = demo_events_table()
    reuse = (
        events.groupby("ga_session_id")["user_pseudo_id"].nunique().rename("distinct_users")
    )
    composite_sizes = events.groupby(["user_pseudo_id", "ga_session_id"]).size()
    return pd.DataFrame(
        {
            "metric": [
                "distinct_user_pseudo_id",
                "distinct_ga_session_id_values",
                "distinct_composite_sessions",
                "ga_session_id_values_used_by_multiple_users",
                "share_session_id_reuse_pct",
                "avg_events_per_composite_session",
                "median_events_per_composite_session",
                "max_events_per_composite_session",
            ],
            "value": [
                float(events["user_pseudo_id"].nunique()),
                float(events["ga_session_id"].nunique()),
                float(len(composite_sizes)),
                float(int((reuse > 1).sum())),
                round(float((reuse > 1).mean() * 100), 3),
                round(float(composite_sizes.mean()), 3),
                float(composite_sizes.median()),
                float(composite_sizes.max()),
            ],
            "synthetic_note": "SYNTHETIC DEMO",
        }
    )


session_df = load_or_query("07_session_feasibility.sql", "session feasibility", demo_builder=demo_session_feasibility)
session_df

[blocked] Skipping 'session feasibility'. Query that would run:
------------------------------------------------------------------------------
WITH window_events AS (
   
   
  SELECT
    user_pseudo_id,
    (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS
ga_session_id
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
)
SELECT
  COUNT(*) AS total_events,
  COUNTIF(ga_session_id IS NOT NULL) AS events_with_ga_session_id,
... [10 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


### 5.3 Running the local validation library on this run's output

`src/growthscope/quality/` holds the project's data-quality checks. They are pure functions over
pandas frames, covered by the pytest suite in `tests/`, and they return a structured `CheckResult`
(name, pass/fail, severity, observed value, threshold, message, evidence) instead of raising — so
the same check can drive a notebook, a Markdown audit table, or a CI gate.

They operate on **aggregates**, not on the raw export. Counting NULLs across millions of nested
event rows is what SQL is for; these checks answer the narrower questions that remain once an
aggregate exists.

Two very different checks are demonstrated here:

1. **Date coverage** — run against the daily series this notebook actually produced. A missing
   export shard appears as a missing day, not as a zero, and is therefore invisible unless it is
   checked explicitly.
2. **Ordered-funnel monotonicity** — a correctly built ordered funnel counts *nested* session sets,
   so step counts can only decrease. A violation means a broken join or a wrong ordering rule, and
   it produces conversion rates above 100 %. That invariant belongs to `int_session_funnel` in
   Sprint 02, so it is demonstrated now on clearly labelled synthetic funnels — including a
   deliberately broken one, because documenting the failure mode is the reason the check exists.

> [!NOTE]
> A failing check is not automatically a defect. Several checks in this library are designed to
> **fail on expected GA4 behaviour** — session-id reuse across users is expected, and reporting it
> is what stops the composite session key from ever being forgotten. Each result carries its own
> message explaining which case it is.

In [10]:
# ---------------------------------------------------------------------------
# Apply the project quality checks to the frames this run actually produced
# ---------------------------------------------------------------------------
def render(results) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "check": r.name,
                "passed": r.passed,
                "severity": r.severity.value,
                "observed": r.metric_value,
                "threshold": r.threshold,
            }
            for r in results
        ]
    )


check_results = []

try:
    from growthscope.quality import check_date_coverage, check_funnel_monotonicity

    # (a) Coverage of the daily series produced above. Skipped when nothing was produced:
    #     an empty series is "no evidence", which is not the same as "a coverage failure".
    if is_empty(daily_df):
        print("date-coverage check skipped: this run produced no daily series (see section 3).")
    else:
        coverage = check_date_coverage(daily_df, "event_date", START_DATE, END_DATE)
        check_results.append(coverage)
        print(f"date coverage       : {coverage.message}")

    # (b) Ordered-funnel invariant, demonstrated on labelled SYNTHETIC funnels only.
    FUNNEL_STEP_COLUMNS = ["s1_view_item", "s2_add_to_cart", "s3_begin_checkout", "s4_purchase"]

    synthetic_healthy_funnel = pd.DataFrame(
        {
            "segment": ["SYNTHETIC - healthy"],
            "s1_view_item": [1000],
            "s2_add_to_cart": [250],
            "s3_begin_checkout": [120],
            "s4_purchase": [40],
        }
    )
    synthetic_broken_funnel = pd.DataFrame(
        {
            "segment": ["SYNTHETIC - deliberately broken"],
            "s1_view_item": [1000],
            "s2_add_to_cart": [250],
            "s3_begin_checkout": [310],  # larger than the previous step: impossible in a nested funnel
            "s4_purchase": [40],
        }
    )

    for frame in (synthetic_healthy_funnel, synthetic_broken_funnel):
        result = check_funnel_monotonicity(frame, FUNNEL_STEP_COLUMNS)
        check_results.append(result)
        print(f"funnel monotonicity : {result.message}")

except ImportError as exc:
    print(f"Quality library unavailable ({exc}).")
    print("Install the project dependencies (`pip install -r requirements.txt`) and re-run.")

print()
print("Quality checks executed:", len(check_results))
render(check_results) if check_results else None

date-coverage check skipped: this run produced no daily series (see section 3).
funnel monotonicity : Funnel steps s1_view_item > s2_add_to_cart > s3_begin_checkout > s4_purchase are non-increasing for all 1 row(s): the steps are properly nested, so conversion rates cannot exceed 100%.
funnel monotonicity : 1 funnel row(s) grow instead of shrink inside an ordered session funnel (1 illegal step comparison(s)). Sessions reaching a later step are a subset of those reaching the previous step, so this can only come from how the funnel was built (LEFT JOIN fan-out, non-nested step filters, steps ordered by the wrong timestamp) and it would show up as a conversion rate above 100%.

Quality checks executed: 2


,check,passed,severity,observed,threshold
0,funnel_monotonicity,True,informational,0,0
1,funnel_monotonicity,False,high,1,0


The quality library intentionally returns a **structured result** rather than raising, so the same
checks can be surfaced in a report, a CI job or a dashboard without rewriting them. Its severities
distinguish three different things that are easily conflated:

| Situation | Correct classification |
|---|---|
| A user declined cookies, so no `user_pseudo_id` exists | **Analytical limitation** — measure and disclose the rate |
| A `transaction_id` appears twice with identical payload | **Duplicate risk** — de-duplicate before revenue math |
| A daily series has a calendar gap | **Coverage defect** — investigate the missing export |
| A funnel step count exceeds the previous step | **Logic defect** — a broken join or ordering rule |

## 6. E-commerce events

### 6.1 Funnel event availability

Query: `sql/exploration/05_funnel_event_audit.sql`.

This answers one question only: **do the events a purchase funnel needs actually exist in this
window, and are they common enough to be measurable?**

It deliberately does **not** produce conversion rates. Presence of `begin_checkout` and
`purchase` events is a prerequisite for a funnel, not a funnel. A conversion rate additionally
requires:

- sessions as the denominator (not events),
- ordering in time (a step only counts if it happened after the previous one),
- de-duplication within a session (multiple `add_to_cart` events = one session step).

Those rules belong to Sprint 02 (`int_sessions`, `int_session_funnel`).

In [11]:
FUNNEL_SEQUENCE = [
    "session_start",
    "view_item",
    "view_item_list",
    "view_cart",
    "add_to_cart",
    "begin_checkout",
    "add_shipping_info",
    "add_payment_info",
    "purchase",
    "refund",
]


def demo_funnel_audit() -> pd.DataFrame:
    events = demo_events_table()
    rows = []
    for position, name in enumerate(FUNNEL_SEQUENCE):
        subset = events[events["event_name"] == name]
        rows.append(
            {
                "event_name": name,
                "sequence_position": position,
                "event_count": int(len(subset)),
                "distinct_users": int(subset["user_pseudo_id"].nunique()),
                "distinct_sessions": int(subset.groupby(["user_pseudo_id", "ga_session_id"]).ngroups),
                "observed_in_window": bool(len(subset) > 0),
                "synthetic_note": "SYNTHETIC DEMO",
            }
        )
    return pd.DataFrame(rows)


funnel_df = load_or_query("05_funnel_event_audit.sql", "funnel event audit", demo_builder=demo_funnel_audit)
funnel_df

[blocked] Skipping 'funnel event audit'. Query that would run:
------------------------------------------------------------------------------
WITH expected_events AS (
   
   
   
  SELECT 1 AS step_order, 'session_start' AS event_name
  UNION ALL SELECT 2, 'first_visit'
  UNION ALL SELECT 3, 'page_view'
  UNION ALL SELECT 4, 'view_item_list'
  UNION ALL SELECT 5, 'view_item'
  UNION ALL SELECT 6, 'add_to_cart'
  UNION ALL SELECT 7, 'remove_from_cart'
  UNION ALL SELECT 8, 'view_cart'
  UNION ALL SELECT 9, 'begin_checkout'
  UNION ALL SELECT 10, 'add_shipping_info'
... [64 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


### 6.2 Purchase and revenue field availability

Query: `sql/exploration/06_purchase_revenue_audit.sql`.

Revenue is the field family where a plausible-looking query produces the wrong number most easily.
Three traps, all structural:

**Trap 1 — two revenue fields, two units.** `ecommerce.purchase_revenue` is in the store's **local
currency**; `ecommerce.purchase_revenue_in_usd` is the currency-normalised value. They are not two
names for one number. Summing `purchase_revenue` across a window that contains more than one
currency adds dollars to euros and yields a meaningless total, which is why
`purchase_revenue_in_usd` is the standing choice for revenue math.

A related claim is worth stating carefully, because it is repeated everywhere and is **not**
confirmed: that `purchase_revenue` is populated *only* when the event currency matches the property
currency. The documentation review for this project could not support it and recorded it as
`[UNRESOLVED]` (`docs/dataset_research.md` §10.2). It is therefore treated as an untested hypothesis
with an explicit test — query 2 of `06_purchase_revenue_audit.sql` reports the NULL rate of each
revenue field separately. If an asymmetry shows up there, it becomes an **observed finding with a
query as evidence**; it never becomes a documented fact by repetition.

**Trap 2 — `purchase` can fire more than once per order.** Client-side and server-side tagging,
retries, and multi-device journeys all produce repeated `purchase` events carrying the same
`transaction_id`. Summing revenue over purchase *events* over-counts. The correct order grain is
`transaction_id`, de-duplicated.

**Trap 3 — items are a repeated record.** `items` is an array. Joining it directly to events
multiplies rows, so item-level revenue must be aggregated *inside* the array, not after a fan-out.

In [12]:
def demo_purchase_audit() -> pd.DataFrame:
    events = demo_events_table()
    purchases = events[events["event_name"] == "purchase"].copy()
    n = len(purchases)
    # Synthetic only. This deliberately models the CONTESTED pattern (purchase_revenue absent
    # while purchase_revenue_in_usd is present) so that the corresponding quality check has
    # something to detect here. Modelling it does not assert it: on real data the pattern is
    # [UNRESOLVED] until 06_purchase_revenue_audit.sql query 2 measures it.
    mismatch = DEMO_RNG.random(n) < 0.25
    duplicated = DEMO_RNG.random(n) < 0.02
    transaction_ids = [f"SYNTH-TXN-{i:05d}" for i in DEMO_RNG.integers(0, max(n - 1, 1), size=n)]
    for idx in np.flatnonzero(duplicated):
        if idx + 1 < n:
            transaction_ids[idx + 1] = transaction_ids[idx]
    return pd.DataFrame(
        {
            "metric": [
                "purchase_events",
                "distinct_transaction_id",
                "purchase_events_with_empty_transaction_id",
                "purchase_events_missing_purchase_revenue",
                "share_missing_purchase_revenue_pct",
                "purchase_events_missing_purchase_revenue_in_usd",
                "distinct_currencies",
                "transaction_ids_with_multiple_purchase_events",
                "revenue_overcount_factor_events_vs_orders",
            ],
            "value": [
                float(n),
                float(len(set(transaction_ids))),
                float(mismatch.sum()),
                float(mismatch.sum()),
                round(float(mismatch.mean() * 100), 3),
                0.0,
                1.0,
                float(sum(1 for c in pd.Series(transaction_ids).value_counts() if c > 1)),
                round(n / max(len(set(transaction_ids)), 1), 4),
            ],
            "synthetic_note": "SYNTHETIC DEMO",
        }
    )


purchase_df = load_or_query("06_purchase_revenue_audit.sql", "purchase revenue audit", demo_builder=demo_purchase_audit)
purchase_df

[blocked] Skipping 'purchase revenue audit'. Query that would run:
------------------------------------------------------------------------------
WITH purchase_events AS (
   
   
   
  SELECT
    user_pseudo_id,
    ecommerce.transaction_id AS transaction_id
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    event_name = 'purchase'
    AND _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
)
SELECT
... [32 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


### 6.3 Event parameters

Query: `sql/exploration/04_event_parameters.sql`.

Because `event_params` is a key–value array, parameter *coverage* varies by event and by GA4
release. Before any parameter is used in a metric, it must be shown to exist with a usable value
column: `ga_session_id` lives in `int_value`, while most attribution parameters live in
`string_value`. Extracting the wrong column returns NULL without an error — a silent failure.

> [!NOTE]
> Acquisition scope is a frequent source of error. `traffic_source` on the event row describes
> **first-user acquisition**: where the user originally came from, repeated on every event of that
> user. Session-scoped sources appear on the `session_start` event and in
> `collected_traffic_source`. Reporting first-user acquisition as session attribution is
> incorrect and is called out explicitly in `docs/data_quality_audit.md`.

In [13]:
def demo_event_parameters() -> pd.DataFrame:
    parameter_keys = [
        ("ga_session_id", "int_value", "session_start, and every event in the session"),
        ("ga_session_number", "int_value", "session_start"),
        ("page_location", "string_value", "page_view"),
        ("page_referrer", "string_value", "page_view"),
        ("source", "string_value", "session_start"),
        ("medium", "string_value", "session_start"),
        ("campaign", "string_value", "session_start"),
        ("engagement_time_msec", "int_value", "user_engagement"),
        ("entrances", "int_value", "session_start"),
        ("transaction_id", "string_value", "purchase"),
        ("value", "double_value", "purchase"),
        ("currency", "string_value", "purchase"),
        ("tax", "double_value", "purchase"),
        ("shipping", "double_value", "purchase"),
        ("coupon", "string_value", "purchase"),
        ("payment_type", "string_value", "purchase"),
    ]
    return pd.DataFrame(
        {
            "parameter_key": [k for k, _, _ in parameter_keys],
            "value_column": [v for _, v, _ in parameter_keys],
            "expected_on_events": [e for _, _, e in parameter_keys],
            "coverage_pct": [None] * len(parameter_keys),  # must be measured live
            "verification_status": ["UNVERIFIED - needs BigQuery"] * len(parameter_keys),
        }
    )


parameter_df = load_or_query("04_event_parameters.sql", "event parameter coverage", demo_builder=demo_event_parameters)
parameter_df

[blocked] Skipping 'event parameter coverage'. Query that would run:
------------------------------------------------------------------------------
WITH window_events AS (
  SELECT
    event_name,
    event_params
  FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201103'  
),
exploded_params AS (
   
   
   
  SELECT
... [24 more lines - full query in sql/exploration/]
------------------------------------------------------------------------------


""


## 7. Initial visualisations

Four charts, each answering a question that changes a decision. Nothing decorative.

| Chart | Question it answers |
|---|---|
| Event distribution | Is the funnel measurable, or are the steps too rare to analyse? |
| Daily volume | Is the window stable enough to compare days? |
| Funnel event availability | Which funnel steps exist at all? |
| Missing identifier rates | How much of reality would a user-level metric describe? |

All charts are **suppressed** unless real or explicitly-labelled synthetic data is present.

In [14]:
# ---------------------------------------------------------------------------
# Plot helpers
# ---------------------------------------------------------------------------
PLOT_STYLE = {
    "figure.figsize": (9, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
}

SYNTHETIC_SUFFIX = "  [SYNTHETIC DEMO - NOT REAL DATA]" if IS_SYNTHETIC else ""


def synthetic_stamp(fig) -> None:
    """Make synthetic output impossible to mistake for a real result.

    Placed on the *figure* rather than the axes: an axes-level stamp collides with the
    title on charts with few bars, and an unreadable warning is the same as no warning.
    """
    if IS_SYNTHETIC:
        fig.text(
            0.995,
            0.005,
            "SYNTHETIC DEMO - NOT REAL DATA",
            ha="right",
            va="bottom",
            fontsize=9,
            color="crimson",
            fontweight="bold",
            bbox={"facecolor": "white", "edgecolor": "crimson", "alpha": 0.95, "pad": 3},
        )


def humanise(label: str) -> str:
    """Turn a column-style identifier into a readable chart label.

    ``share_missing_user_pseudo_id_pct`` -> ``missing user pseudo id (%)``
    """
    text = str(label)
    text = re.sub(r"^share_missing_", "", text)
    text = re.sub(r"^share_", "", text)
    text = re.sub(r"^pct_", "", text)
    suffix = " (%)" if text.endswith("_pct") else ""
    text = re.sub(r"_pct$", "", text)
    return text.replace("_", " ").strip() + suffix


def plot_barh(frame, label_col, value_col, title, xlabel, color="#3B6FB6", top=None):
    if not has_rows(frame) or label_col not in frame or value_col not in frame:
        print(f"chart skipped -> '{title}' (no data)")
        return None
    data = frame[[label_col, value_col]].dropna().sort_values(value_col, ascending=True)
    if top:
        data = data.tail(top)

    # A single bar drawn at the default height fills the whole axes and reads as a block of
    # colour rather than a measurement. Shrink the bar and pad the axis when bars are few.
    bar_height = 0.7 if len(data) >= 4 else 0.35

    with plt.rc_context(PLOT_STYLE):
        fig, ax = plt.subplots()
        ax.barh(data[label_col].astype(str), data[value_col].astype(float),
                height=bar_height, color=color)
        if len(data) <= 3:
            ax.margins(y=0.75)
        ax.set_title(title + SYNTHETIC_SUFFIX, loc="left", fontweight="bold")
        ax.set_xlabel(xlabel)
        synthetic_stamp(fig)
        fig.tight_layout()
        plt.show()
    return ax


def plot_line(frame, x_col, y_col, title, xlabel, ylabel, color="#3B6FB6"):
    if not has_rows(frame) or x_col not in frame or y_col not in frame:
        print(f"chart skipped -> '{title}' (no data)")
        return None
    data = frame[[x_col, y_col]].dropna().sort_values(x_col)
    with plt.rc_context(PLOT_STYLE):
        fig, ax = plt.subplots()
        ax.plot(data[x_col].astype(str), data[y_col].astype(float), marker="o", color=color)
        ax.set_title(title + SYNTHETIC_SUFFIX, loc="left", fontweight="bold")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        synthetic_stamp(fig)
        fig.tight_layout()
        plt.show()
    return ax


print(f"Plot helpers ready. Synthetic output: {IS_SYNTHETIC}")

Plot helpers ready. Synthetic output: False


In [15]:
# --- Chart 1: event distribution -------------------------------------------------
plot_barh(
    event_df,
    label_col="event_name",
    value_col="event_count" if "event_count" in event_df else "event_share_pct",
    title="Event distribution in the exploration window",
    xlabel="Events (count, not sessions)",
    top=20,
)

chart skipped -> 'Event distribution in the exploration window' (no data)


In [16]:
# --- Chart 2: daily event volume -------------------------------------------------
event_column = "event_count" if "event_count" in daily_df else None
if event_column:
    plot_line(
        daily_df,
        x_col="event_date",
        y_col=event_column,
        title="Daily event volume",
        xlabel="Date",
        ylabel="Events",
    )
else:
    print("chart skipped -> daily event volume (no data)")

chart skipped -> daily event volume (no data)


In [17]:
# --- Chart 3: funnel event availability -----------------------------------------
plot_barh(
    funnel_df,
    label_col="event_name",
    value_col="distinct_sessions" if "distinct_sessions" in funnel_df else "event_count",
    title="Funnel-relevant events: sessions in which each step occurred",
    xlabel="Distinct (user_pseudo_id, ga_session_id) sessions",
    color="#2E8B57",
)

chart skipped -> 'Funnel-relevant events: sessions in which each step occurred' (no data)


In [18]:
# --- Chart 4: missing identifier rates ------------------------------------------
missing_rows = pd.DataFrame()
if has_rows(user_df):
    if {"metric", "value"}.issubset(user_df.columns):
        missing_rows = user_df[user_df["metric"].astype(str).str.contains("share_missing", na=False)]
    elif {"identifier", "share_missing_pct"}.issubset(user_df.columns):
        missing_rows = user_df.rename(columns={"identifier": "metric", "share_missing_pct": "value"})
if has_rows(missing_rows):
    # Column-style names such as `share_missing_user_pseudo_id_pct` are precise but unreadable
    # on a chart, so they are converted for display only.
    missing_rows = missing_rows.assign(identifier=[humanise(m) for m in missing_rows["metric"]])
    plot_barh(
        missing_rows,
        label_col="identifier",
        value_col="value",
        title="Missing identifier rates",
        xlabel="% of events in the window",
        color="#B4552D",
    )
else:
    print("chart skipped -> missing identifier rates (no data)")

chart skipped -> missing identifier rates (no data)


## 8. Preliminary findings

This section follows one rule: **an observation must be traceable to something that ran.**
The status column states exactly how far each item was verified.

| # | Item | Status | Meaning |
|---|---|---|---|
| F1 | Dataset reference and shard naming pattern `events_YYYYMMDD` | `[DOCUMENTED]` | Confirmed by Google's public documentation, not by a query |
| F2 | Documented coverage Nov 2020 – Jan 2021 | `[DOCUMENTED]` | Exact first/last available day must be confirmed by query |
| F3 | `ga_session_id` is session-scoped and not globally unique | `[DOCUMENTED]` + structural reasoning | Composite key `(user_pseudo_id, ga_session_id)` is mandatory |
| F4 | `purchase_revenue` is the store's local currency while `purchase_revenue_in_usd` is USD, so summing the former across mixed currencies is meaningless | `[DOCUMENTED]` | Forces `purchase_revenue_in_usd` as the primary revenue field |
| F4b | "`purchase_revenue` is NULL when the store currency differs from the property currency" | `[UNRESOLVED]` — the documentation review could not support it (`docs/dataset_research.md` §10.2); tested by `06_purchase_revenue_audit.sql` query 2 | Must be measured, never assumed |
| F5 | `traffic_source` is first-user acquisition, not session attribution | `[DOCUMENTED]` | Acquisition reporting must not mix the two scopes |
| F6 | No advertising cost data exists in the dataset | `[DOCUMENTED]` | CAC / ROAS are out of scope; no substitute will be invented |
| F7 | Actual event counts, user counts, daily volume in the window | `[BLOCKED — no BigQuery access]` | Requires executing `sql/exploration/01..03` |
| F8 | Actual missing-identifier rates | `[BLOCKED — no BigQuery access]` | Requires `sql/exploration/03_user_identification.sql` |
| F9 | Actual purchase de-duplication rate on `transaction_id` | `[BLOCKED — no BigQuery access]` | Requires `sql/exploration/06_purchase_revenue_audit.sql` |
| F10 | Actual session key collision rate (one session id, many users) | `[BLOCKED — no BigQuery access]` | Requires `sql/exploration/07_session_feasibility.sql` |

> [!IMPORTANT]
> Rows F7–F10 are **the empirical core of Sprint 01** and cannot be filled from documentation.
> They are deliberately left blank rather than estimated. Sprint 02 must start by executing those
> four queries and replacing each row with a measured value and its exact window.

In [19]:
# ---------------------------------------------------------------------------
# Machine-generated status of this run
# ---------------------------------------------------------------------------
produced = {
    "shard_inventory": shard_df,
    "event_inventory": event_df,
    "daily_volume": daily_df,
    "user_identification": user_df,
    "session_feasibility": session_df,
    "funnel_audit": funnel_df,
    "purchase_audit": purchase_df,
    "event_parameters": parameter_df,
}

status = pd.DataFrame(
    {
        "section": list(produced),
        "rows_produced": [len(v) if has_rows(v) else 0 for v in produced.values()],
        "source": [MODE] * len(produced),
    }
)

print(f"Run mode        : {MODE}")
print(f"Synthetic data  : {IS_SYNTHETIC}")
print(f"Frames produced : {int((status['rows_produced'] > 0).sum())} / {len(status)}")
print()
print(status.to_string(index=False))
print()

if MODE == "blocked":
    print("No empirical findings were produced in this run. See docs/data_quality_audit.md")
    print("for the exact queries that convert each blocked item into a verified one.")
elif IS_SYNTHETIC:
    print("All rows above are SYNTHETIC. They demonstrate the analysis pipeline only.")
    print("They must never be cited as findings about the GA4 dataset.")
else:
    print("Record the window, the query file and the observed values in docs/ before citing them.")
    print("An unrecorded number is not a finding.")

Run mode        : blocked
Synthetic data  : False
Frames produced : 0 / 8

            section  rows_produced  source
    shard_inventory              0 blocked
    event_inventory              0 blocked
       daily_volume              0 blocked
user_identification              0 blocked
session_feasibility              0 blocked
       funnel_audit              0 blocked
     purchase_audit              0 blocked
   event_parameters              0 blocked

No empirical findings were produced in this run. See docs/data_quality_audit.md
for the exact queries that convert each blocked item into a verified one.


### Hypotheses (explicitly not findings)

The following are reasonable expectations about a GA4 e-commerce store. They are recorded here so
they can be **tested and possibly falsified** in Sprint 02 — not reported as results.

- **H1** — Acquisition is dominated by a small number of sources, with a long tail of low-volume
  referrals; traffic share and purchase share will not rank channels identically.
- **H2** — The largest absolute drop-off will occur in the upper funnel (`view_item` →
  `add_to_cart`), while the largest *percentage* drop-off will occur at `add_to_cart` →
  `begin_checkout`.
- **H3** — Mobile sessions will outnumber desktop sessions but convert at a lower rate.
- **H4** — A non-trivial share of sessions will contain no product-view event at all
  (bounce / non-commercial traffic), which lowers the funnel entry denominator without indicating
  a defect.

Each hypothesis has a stated falsification condition; none may enter the dashboard as a conclusion
without the supporting query and window.

## 9. Data limitations

### Dataset limitations

1. **Obfuscated and partially adjusted.** Google states the sample is obfuscated and adjusted for
   demonstration. Absolute magnitudes are therefore illustrative; the *structure* and the
   *relative* patterns are what the project studies.
2. **No advertising cost.** There is no cost table. CAC, ROAS, ad profit and channel
   profitability cannot be computed. This is a missing external dataset, not a modelling choice.
3. **Fixed historical window.** A ~3-month window in late 2020 cannot support seasonality
   analysis, year-over-year comparison, or long-horizon retention. A cohort observed for three
   months has, at most, a three-month retention curve.
4. **Cookie-scoped identities.** `user_pseudo_id` is a browser/device identifier. Cross-device
   journeys look like multiple users. `user_id` is empty for essentially the whole dataset, so no
   cross-device stitching is available.
5. **Sample size limits detectable effects.** The public sample is small. Any comparison of
   conversion rates between segments needs the segment's denominator stated, and small segments
   should not be reported as differences.
6. **Attribution ambiguity.** No multi-touch attribution exists in the export. First-user
   acquisition is available; per-session source is partially available. Neither is a causal
   measurement of marketing effectiveness.

### Environment limitations (this sprint)

7. **Live BigQuery execution was not performed.** No Google Cloud credentials were available in
   the build environment (the BigQuery API returned HTTP 401). Every statistic that requires data
   is marked `[BLOCKED — no BigQuery access]` and left blank. The queries and the notebook are
   ready; only credentials are missing.

### Claims explicitly out of scope

- Anything causal ("channel X caused a 12 % lift").
- Anything predictive or budget-optimising.
- Ad profitability of any kind.
- Generalisation from this demo store to other e-commerce businesses.

## 10. Next steps

### Immediate: convert blocked findings into verified ones

| Step | Action | Artefact |
|---|---|---|
| 1 | Obtain BigQuery Sandbox access (free, no credit card) | documented in `README.md` |
| 2 | Dry-run every query in `sql/exploration/` and record bytes processed | `sql/exploration/README.md` |
| 3 | Execute `00`–`07` over `2020-11-01` → `2020-11-03` | verified result set |
| 4 | Re-run this notebook in `live` mode and record the window and values | updated findings table |
| 5 | Update `docs/data_quality_audit.md`, replacing `[BLOCKED]` with measured findings | audit revision |

### Sprint 02 data models

Scope discipline: Sprint 02 builds the model of a session, not the dashboard.

| Model | Grain | Purpose | Key rules |
|---|---|---|---|
| `stg_ga4_events` | one row per event | typed, de-nested staging layer | Unpack `event_params` once, cast microseconds to timestamp, keep `event_date` for partitioning |
| `int_sessions` | one row per `(user_pseudo_id, ga_session_id)` | session reconstruction | Session start/end from `event_timestamp`, event counts, session-level pageviews, session-scoped traffic source |
| `int_session_funnel` | one row per session | funnel flags | Step flags plus *ordered* step achievement using window functions over `event_timestamp` |
| `fct_purchases` | one row per de-duplicated `transaction_id` | revenue | De-duplicate on `transaction_id`; use `purchase_revenue_in_usd`; aggregate `items` inside the array |
| `fct_orders_items` | one row per item line | product analytics | Unnest `items` **after** purchase de-duplication to avoid double counting |
| `mart_daily_kpis` | one row per day | executive metrics | Revenue, orders, sessions, AOV, purchase conversion — with denominators explicitly stated |

### Definition of done for a metric before it reaches a dashboard

1. Formula written in `docs/metric_dictionary.md` with numerator and denominator in words.
2. Implemented in SQL against a named window.
3. Cross-checked with an independent second implementation or a hand calculation on a known subset.
4. Null/missing behaviour documented (what happens when a field is absent).
5. Validation status raised from `PROPOSED` to `VERIFIED (LIVE)`.

Anything that fails step 3 does not reach the dashboard. That is the difference between an
analytics project and a chart generator.

In [20]:
# ---------------------------------------------------------------------------
# Optional: export aggregates for the SAMPLE mode
# ---------------------------------------------------------------------------
# Enabled with GROWTHSCOPE_EXPORT_SAMPLE=1, and only from a live run. Aggregated result sets are
# small and contain no user-level rows, so they are safe to commit; raw events never are.
if EXPORT_SAMPLE and MODE == "live":
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    exported = {
        "01_event_inventory": event_df,
        "02_daily_event_volume": daily_df,
        "03_user_identification": user_df,
        "04_event_parameters": parameter_df,
        "05_funnel_event_audit": funnel_df,
        "06_purchase_revenue_audit": purchase_df,
        "07_session_feasibility": session_df,
    }
    for name, frame in exported.items():
        if has_rows(frame):
            target = SAMPLE_DIR / f"{name}.csv"
            frame.to_csv(target, index=False)
            print(f"wrote {target.relative_to(REPO_ROOT)} ({len(frame)} rows)")
elif EXPORT_SAMPLE:
    print(f"Export skipped: GROWTHSCOPE_EXPORT_SAMPLE=1 requires live mode (current mode: {MODE}).")
else:
    print("Export disabled. Set GROWTHSCOPE_EXPORT_SAMPLE=1 in live mode to refresh data/sample/.")

Export disabled. Set GROWTHSCOPE_EXPORT_SAMPLE=1 in live mode to refresh data/sample/.

---

### Reproducibility record

| Item | Value |
|---|---|
| Repository | <https://github.com/nnurkass/growthscope> |
| Sprint | 01 — Project Foundation & Data Discovery |
| Window used | `GA4_START_DATE` → `GA4_END_DATE` (default `2020-11-01` → `2020-11-03`) |
| Cost guardrail | `maximum_bytes_billed = BQ_MAX_BYTES_BILLED` (default 2 GB) |
| Queries | `sql/exploration/00..07_*.sql` |
| Local checks | `src/growthscope/quality/` · `tests/` · `python -m pytest` |
| **Mode of the outputs saved in this file** | **`blocked`** — no Google Cloud credentials existed in the build environment, so the saved outputs are the honest "what would run" trace, not results |

> [!IMPORTANT]
> The outputs committed inside this `.ipynb` were produced by a `blocked` run. They show which
> queries would execute and confirm that the notebook itself is error-free — nothing more. Creating
> this file deliberately did **not** involve pasting invented numbers into it.

**How to reproduce a verified run**

```bash
# 1. environment
python -m venv .venv
.venv\Scripts\activate            # Windows PowerShell
# source .venv/bin/activate        # macOS / Linux
pip install -r requirements.txt

# 2. credentials (BigQuery Sandbox is free and needs no credit card)
gcloud auth application-default login
export GOOGLE_CLOUD_PROJECT=your-project-id     # Windows: $env:GOOGLE_CLOUD_PROJECT="your-project-id"

# 3. execute
jupyter lab notebooks/01_data_discovery.ipynb
python -m pytest
```

> [!NOTE]
> If a number appears anywhere in GrowthScope without a window, a query reference and a validation
> status, it is not a result yet.